In [27]:
import pandas as pd
import numpy as np
import os
import json
import torch.nn as nn
import os
import json
import pandas as pd
import torch
import torchaudio
import torchaudio.transforms as T
import soundfile as sf
from datetime import datetime
from tqdm.auto import tqdm
import time as time
from tqdm import tqdm
import re


In [16]:
# file_dir = "/home/wangan/아동청소년상담데이터Sample/02.라벨링데이터/out"
# dir=os.listdir(file_dir)
# real_path=[os.path.join(file_dir, p) for p in dir]
# real_path

In [28]:
def is_valid_time_format(time_str):
    return re.match(r'\d{2}:\d{2}\.\d{3}', time_str) is not None

def time_to_seconds(time_str):
    time_obj = datetime.strptime(time_str, '%M:%S.%f')
    seconds = time_obj.minute * 60 + time_obj.second + time_obj.microsecond / 1e6
    return seconds

def parsing_audio(file_dir, wav_dir, save_new_dir):
    
    dir = os.listdir(file_dir)
    real_path=[os.path.join(file_dir, p) for p in dir]

    for file in tqdm(real_path):
        
        # json load
        with open(file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        
        wav_name = os.path.splitext(os.path.basename(file))[0]
        waveform, sample_rate = torchaudio.load(os.path.join(wav_dir, f"{wav_name}.mp3"))


        times = []
        j = 0
        for item in data.get("list", []):
            j += 1
            for sub_item in item.get("list", []):
                for audio in sub_item.get("audio", []):
                    if audio['type'] == 'A':
                        times.append({"start": audio["start"], "end": audio["end"]})
                merged_audio = None
                if len(times) == 0:
                    continue
                for k in range(len(times)):
                    start_str = times[k]["start"]
                    end_str = times[k]["end"]
                    if not is_valid_time_format(start_str) or not is_valid_time_format(end_str):
                        continue
                    
                    start_time = time_to_seconds(start_str)
                    end_time = time_to_seconds(end_str)
                    
                    start_sample = int(start_time * sample_rate)
                    end_sample = int(end_time * sample_rate)
                    trimmed_waveform = waveform[:, start_sample:end_sample]

                    if merged_audio is None:
                        merged_audio = trimmed_waveform
                    else:
                        merged_audio = torch.cat((merged_audio, trimmed_waveform), dim=1)
                #import pdb;pdb.set_trace()
                sf.write(os.path.join(save_new_dir, f"{wav_name}-{j}.mp3"), merged_audio[0].numpy(), sample_rate)
                


In [29]:
#훈련 데이터 파싱

file_dir = "/home/wangan/Data/Training/02/" #json파일
wav_dir = "/home/wangan/Data/Training/jihwan_local_train_audio/" #음성 파일
save_new_dir = "/home/wangan/Data/Training/preprocessed_audio_train/" 
parsing_audio(file_dir, wav_dir, save_new_dir)

100%|██████████| 2876/2876 [18:14:51<00:00, 22.84s/it]   


In [31]:
#테스트 데이터 파싱

file_dir = "/home/wangan/Data/Validation/02/" #json파일
wav_dir = "/home/wangan/Data/Validation/jihwan_local_test_audio/" #음성 파일
save_new_dir = "/home/wangan/Data/Validation/preprocessed_audio_test/" 
parsing_audio(file_dir, wav_dir, save_new_dir)

100%|██████████| 360/360 [2:17:07<00:00, 22.85s/it]  


In [36]:
train = "/home/wangan/Data/Training/preprocessed_audio_train"
train_processed_count = len(os.listdir(train))
train_origin_count = train_processed_count / 7


print('훈련데이터 전처리 파일 개수',train_processed_count,'개')
print('훈련데이터 원본 개수',train_origin_count,'개')

훈련데이터 전처리 파일 개수 20132 개
훈련데이터 원본 개수 2876.0 개


In [37]:
test = "/home/wangan/Data/Validation/preprocessed_audio_test"
test_processed_count = len(os.listdir(test))
test_origin_count = test_processed_count / 7


print('테스트데이터 전처리한 파일 개수',test_processed_count,'개')
print('테스트데이터 원본 개수',test_origin_count,'개')



테스트데이터 전처리한 파일 개수 2520 개
테스트데이터 원본 개수 360.0 개


In [26]:
wave_dir = "/home/wangan/Data/Training/jihwan_local_train_audio/"
file = '/home/wangan/Data/Training/02/0223.json'

In [17]:
def is_valid_time_format(time_str):
    return re.match(r'\d{2}:\d{2}\.\d{3}', time_str) is not None

def time_to_seconds(time_str):
    time_obj = datetime.strptime(time_str, '%M:%S.%f')
    seconds = time_obj.minute * 60 + time_obj.second + time_obj.microsecond / 1e6
    return seconds

# json load
with open(file, 'r', encoding='utf-8') as f:
    data = json.load(f)


wav_name = os.path.splitext(os.path.basename(file))[0]
waveform, sample_rate = torchaudio.load(os.path.join(wav_dir, f"{wav_name}.mp3"))


times = []
j = 0
for item in data.get("list", []):
    j += 1
    for sub_item in item.get("list", []):
        for audio in sub_item.get("audio", []):
            if audio['type'] == 'A':
                times.append({"start": audio["start"], "end": audio["end"]})
        merged_audio = None
        if len(times) == 0:
            continue
        for k in range(len(times)):
            start_str = times[k]["start"]
            end_str = times[k]["end"]
            if not is_valid_time_format(start_str) or not is_valid_time_format(end_str):
                continue
            
            start_time = time_to_seconds(start_str)
            end_time = time_to_seconds(end_str)
            
            start_sample = int(start_time * sample_rate)
            end_sample = int(end_time * sample_rate)
            trimmed_waveform = waveform[:, start_sample:end_sample]

            if merged_audio is None:
                merged_audio = trimmed_waveform
            else:
                merged_audio = torch.cat((merged_audio, trimmed_waveform), dim=1)
        
# print(times)
        # print("ok")
        sf.write(os.path.join(save_new_dir, f"{wav_name}-{j}.mp3"), merged_audio[0].numpy(), sample_rate)